In [ ]:
%%writefile /kaggle/working/chbmit_cnnlstm.py
import os
import re
import torch
import mne
import numpy as np
from scipy.signal import firwin, lfilter

CANONICAL_CHANNELS = [
    'FP1-F7','F7-T7','T7-P7','P7-O1',
    'FP1-F3','F3-C3','C3-P3','P3-O1',
    'FZ-CZ','CZ-PZ',
    'FP2-F4','F4-C4','C4-P4','P4-O2',
    'FP2-F8','F8-T8','T8-P8','P8-O2',
    'P7-T7','T7-FT9','FT9-FT10','FT10-T8',
]

class SeizurePreprocessing:
    def __init__(self, root_dir, save_dir, window_sec=5, sfreq=256,
                 min_preictal_min=5, target_channels=None):
        self.root_dir         = root_dir
        self.save_dir         = save_dir
        self.sfreq            = sfreq
        self.window_size      = window_sec * sfreq      # 5 * 256 = 1280 samples
        self.min_preictal_sec = min_preictal_min * 60
        self.target_channels  = target_channels
        os.makedirs(save_dir, exist_ok=True)

    def apply_broadband_filter(self, data):
        taps = firwin(101, [0.5, 40.0], pass_zero=False, fs=self.sfreq)
        return lfilter(taps, 1.0, data, axis=-1)

    @staticmethod
    def instance_normalise(window):
        mean = window.mean(axis=1, keepdims=True)
        std  = window.std(axis=1,  keepdims=True) + 1e-8
        return (window - mean) / std

    def extract_segments(self, patient_id, edf_file,
                         start_time=None, mode='interictal', max_segments=None):
        edf_path = os.path.join(self.root_dir, patient_id, edf_file)
        if not os.path.exists(edf_path):
            return 0

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        if int(raw.info["sfreq"]) != self.sfreq:
            raw.resample(self.sfreq)

        # Build channel matrix — zero-fill missing canonical channels
        data_list = []
        for ch in self.target_channels:
            if ch in raw.ch_names:
                data_list.append(raw.get_data(picks=[ch]))
            else:
                data_list.append(np.zeros((1, len(raw.times))))

        data = self.apply_broadband_filter(np.vstack(data_list))
        # data shape: (22, T)

        if mode == 'preictal' and start_time is not None:
            e_idx        = int(start_time * self.sfreq)
            s_idx        = max(0, int((start_time - 3600) * self.sfreq))
            segment_data = data[:, s_idx:e_idx]
        else:
            segment_data = data

        step_size = int(2 * self.sfreq) if mode == 'preictal' else self.window_size
        n_windows = max(0, (segment_data.shape[1] - self.window_size) // step_size + 1)
        extracted = 0

        for i in range(n_windows):
            if max_segments and extracted >= max_segments:
                break

            start_w = i * step_size
            window  = segment_data[:, start_w: start_w + self.window_size]
            if window.shape[1] < self.window_size:
                continue

            window = self.instance_normalise(window)
            # window shape: (22, 1280)

            # Append average reference channel → final shape (23, 1280)
            # EEGNet depthwise conv uses n_channels=23 as its spatial dim
            avg_channel = np.mean(window, axis=0, keepdims=True)   # (1, 1280)
            window      = np.vstack([window, avg_channel])          # (23, 1280)

            save_path = os.path.join(
                self.save_dir,
                f"{mode}_{patient_id}_{edf_file}_win{i}.pt"
            )
            torch.save({
                "data"      : torch.tensor(window, dtype=torch.float32),
                "label"     : 1 if mode == 'preictal' else 0,
                "patient_id": patient_id,
            }, save_path)
            extracted += 1

        return extracted


def auto_get_seizure_times(subject_dir):
    seizure_map   = {}
    summary_files = [f for f in os.listdir(subject_dir) if "summary.txt" in f.lower()]
    if not summary_files:
        return seizure_map
    with open(os.path.join(subject_dir, summary_files[0]), "r") as f:
        content = f.read()
    blocks = re.split(r"File Name:\s*", content, flags=re.IGNORECASE)
    for block in blocks[1:]:
        lines = block.splitlines()
        if not lines:
            continue
        file_name = lines[0].strip()
        starts = re.findall(r"Seizure.*?Start.*?Time:\s*(\d+)\s*seconds", block, re.I)
        ends   = re.findall(r"Seizure.*?End.*?Time:\s*(\d+)\s*seconds",   block, re.I)
        seizure_map[file_name] = {
            "starts": [int(s) for s in starts],
            "ends"  : [int(e) for e in ends],
        }
    return seizure_map

In [ ]:
import os
import random
import importlib
import chbmit_cnnlstm as eeg
importlib.reload(eeg)

ROOT_DIR = (
    "/kaggle/input/datasets/abhishekinnvonix/"
    "seizure-epilepcy-chb-mit-eeg-dataset-pediatric/"
    "chb-mit-scalp-eeg-database-1.0.0"
)
SAVE_DIR = "/kaggle/temp/data"
os.makedirs(SAVE_DIR, exist_ok=True)

PATIENTS = [
    "chb01", "chb02", "chb03", "chb04", "chb05",
    "chb07", "chb09", "chb10", "chb14", "chb18",
    "chb20", "chb21", "chb22", "chb23",
]

preprocessor = eeg.SeizurePreprocessing(
    root_dir         = ROOT_DIR,
    save_dir         = SAVE_DIR,
    window_sec       = 5,           # → 1280 timepoints at 256 Hz (matches CFG)
    sfreq            = 256,
    min_preictal_min = 5,
    target_channels  = list(eeg.CANONICAL_CHANNELS),
)

print("\n--- CHB-MIT Dataset Generation (EEGNet+BiLSTM) ---")
print(f"  Window  : {preprocessor.window_size} samples "
      f"({preprocessor.window_size // preprocessor.sfreq}s @ {preprocessor.sfreq}Hz)")
print(f"  Channels: 22 canonical + 1 avg reference = 23 total\n")

total_pre, total_inter = 0, 0

for p_id in PATIENTS:
    p_path = os.path.join(ROOT_DIR, p_id)
    if not os.path.isdir(p_path):
        print(f"  [SKIP] {p_id} — directory not found")
        continue

    seizure_info  = eeg.auto_get_seizure_times(p_path)
    pre_count     = 0
    seizure_files = [f for f, t in seizure_info.items() if t["starts"]]

    # ---- Preictal segments ----
    for file_name in seizure_files:
        for t_start in seizure_info[file_name]["starts"]:
            added = preprocessor.extract_segments(
                p_id, file_name, start_time=t_start, mode="preictal"
            )
            pre_count += added

    # ---- Interictal segments (balanced to preictal count) ----
    inter_count  = 0
    normal_files = [f for f, t in seizure_info.items() if not t["starts"]]
    random.shuffle(normal_files)

    for file_name in normal_files:
        if inter_count >= pre_count:
            break
        added = preprocessor.extract_segments(
            p_id, file_name, mode="interictal",
            max_segments=(pre_count - inter_count)
        )
        inter_count += added

    total_pre   += pre_count
    total_inter += inter_count
    print(f"  {p_id}: {pre_count:5d} Preictal | {inter_count:5d} Interictal")

print(f"\n{'─'*45}")
print(f"  TOTAL : {total_pre:6d} Preictal | {total_inter:6d} Interictal")
print(f"  GRAND : {total_pre + total_inter:6d} windows")
print(f"  Shape per window: (23, 1280) — float32")
print(f"{'─'*45}")

In [ ]:
import math
import os, re, copy, random, torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

ALL_PATIENTS = [
    "chb01", "chb02", "chb03", "chb04", "chb05",
    "chb07", "chb09", "chb10", "chb14", "chb18",
    "chb20", "chb21", "chb22", "chb23",
]

CFG = {
    "data_path"       : "/kaggle/temp/data",
    "batch_size"      : 64,
    "effective_batch" : 256,
    "lr"              : 1e-3,
    "max_epochs"      : 40,
    "patience"        : 7,
    "n_channels"      : 23,
    "n_timepoints"    : 1280,
    "F1"              : 8,
    "D"               : 2,
    "F2"              : 16,
    "dropout"         : 0.5,
    "lstm_hidden"     : 128,
    "lstm_layers"     : 2,
    "lstm_dropout"    : 0.3,
    "threshold"       : 0.50,
    "vote_window"     : 120,
    "vote_thresh"     : 0.40,
    "vote_step"       : 12,
    "window_sec"      : 5,
    "merge_gap_sec"   : 30,
    "kd_epochs"       : 15,
    "kd_support_ratio": 0.15,
    "T_temp"          : 4.0,
    "attn_heads"      : 4,
    "attn_dim"        : 16,
    "attn_dropout"    : 0.1,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ==========================================
# ATTENTION
# ==========================================
class TemporalAttention(nn.Module):
    """
    Multi-head causal self-attention over EEGNet's temporal output.
    Each position can only attend to itself and past positions,
    preserving causal structure for real-time streaming use.
    """
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.scale   = math.sqrt(self.d_head)

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model,     bias=False)
        self.norm      = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D  = x.shape
        residual = x

        qkv      = self.qkv_proj(x)
        q, k, v  = qkv.chunk(3, dim=-1)

        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        attn = torch.matmul(q, k.transpose(-2, -1)) / self.scale
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.out_proj(out)

        return self.norm(out + residual)


# ==========================================
# MODEL
# ==========================================
class EEGNetBiLSTM(nn.Module):
    """
    EEGNet (spatial-temporal front-end)
      → Linear projection (F2 → attn_dim)
      → TemporalAttention (causal multi-head self-attention)
      → Causal LSTM
      → MLP classifier
    """
    def __init__(self, n_channels=23, n_timepoints=1280, F1=8, D=2,
                 dropout=0.5, lstm_hidden=128, lstm_layers=2,
                 lstm_dropout=0.3, n_classes=2,
                 attn_heads=4, attn_dim=16, attn_dropout=0.1):
        super().__init__()
        F2 = F1 * D

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, kernel_size=(1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )

        self.attn_proj = nn.Linear(F2, attn_dim, bias=False)

        self.attention = TemporalAttention(
            d_model = attn_dim,
            n_heads = attn_heads,
            dropout = attn_dropout,
        )

        self.bilstm = nn.LSTM(
            input_size    = attn_dim,
            hidden_size   = lstm_hidden,
            num_layers    = lstm_layers,
            batch_first   = True,
            bidirectional = False,
            dropout = lstm_dropout if lstm_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 64),
            nn.ELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        B, C, T = x.shape
        x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.squeeze(2).permute(0, 2, 1)    # (B, T', F2)
        x = self.attn_proj(x)                # (B, T', attn_dim)
        x = self.attention(x)                # (B, T', attn_dim)
        x, _ = self.bilstm(x)               # (B, T', lstm_hidden)
        x = x[:, -1, :]                     # (B, lstm_hidden)
        return self.classifier(x)


# ==========================================
# DATASET
# ==========================================
class EEGDataset(Dataset):
    def __init__(self, file_paths):
        self.files  = file_paths
        self.labels = np.array([1 if "preictal" in f else 0 for f in file_paths])

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        obj = torch.load(self.files[idx], weights_only=True)
        return obj["data"], torch.tensor(obj["label"], dtype=torch.long)


def make_weighted_sampler(labels):
    counts  = np.bincount(labels)
    weights = 1.0 / counts[labels]
    return WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.float32),
        num_samples=len(labels), replacement=True
    )


# ==========================================
# TEMPORAL VOTING
# ==========================================
def temporal_voting(test_files, probs, cfg):
    records = []
    for i, fpath in enumerate(test_files):
        fname = os.path.basename(fpath)
        m = re.match(r"(interictal|preictal)_(chb\d+)_(.+?)_win(\d+)\.pt", fname)
        if m:
            mode = m.group(1)
            key  = m.group(3)
            win  = int(m.group(4))
        else:
            parts = fname.rsplit("_win", 1)
            key   = parts[0]
            win   = int(parts[1].replace(".pt", "")) if len(parts) == 2 else i
            mode  = "preictal" if "preictal" in fname else "interictal"

        records.append({
            "key" : key,
            "idx" : win,
            "prob": probs[i],
            "true": 1 if mode == "preictal" else 0,
        })

    records.sort(key=lambda r: (r["key"], r["idx"]))
    groups = defaultdict(list)
    for r in records:
        groups[r["key"]].append(r)

    all_true, all_pred, all_probs_ord = [], [], []
    vw, vs, vt = cfg["vote_window"], cfg["vote_step"], cfg["vote_thresh"]

    for key, wins in groups.items():
        n      = len(wins)
        probs_ = np.array([w["prob"] for w in wins])
        preds  = (probs_ > cfg["threshold"]).astype(int)
        trues  = np.array([w["true"] for w in wins])
        alarms = np.zeros(n, dtype=int)

        for start in range(0, max(1, n - vw + 1), vs):
            block = preds[start: start + vw]
            if len(block) > 0 and block.mean() >= vt:
                alarms[start: start + vw] = 1

        all_pred.extend(alarms)
        all_true.extend(trues)
        all_probs_ord.extend(probs_)

    return np.array(all_true), np.array(all_pred), np.array(all_probs_ord)


def count_alarm_events(pred_array, true_array, window_sec=5, merge_gap_sec=30):
    merge_gap_windows = int(merge_gap_sec / window_sec)

    def extract_events(pred):
        events, in_alarm, gap_count, start = [], False, 0, 0
        for i, p in enumerate(pred):
            if p == 1:
                if not in_alarm:
                    start, in_alarm = i, True
                gap_count = 0
            else:
                if in_alarm:
                    gap_count += 1
                    if gap_count > merge_gap_windows:
                        events.append((start, i - gap_count))
                        in_alarm, gap_count = False, 0
        if in_alarm:
            events.append((start, len(pred) - 1))
        return events

    alarm_events = extract_events(pred_array)
    true_alarms  = sum(1 for s, e in alarm_events if any(true_array[s:e+1] == 1))
    false_alarms = len(alarm_events) - true_alarms
    inter_hours  = float(np.sum(true_array == 0)) * window_sec / 3600.0
    total_hours  = float(len(true_array))          * window_sec / 3600.0
    sensitivity  = true_alarms / max(1, true_alarms + false_alarms)
    return true_alarms, false_alarms, inter_hours, total_hours, sensitivity


# ==========================================
# PHASE 1: TRAIN ON N-1 PATIENTS
# ==========================================
def train_phase1(train_paths, test_patient_id, cfg):
    all_files = [os.path.join(cfg["data_path"], f)
                 for f in os.listdir(cfg["data_path"]) if f.endswith(".pt")]

    train_paths = [f for f in all_files if test_patient_id not in f]
    train_sub, val_sub = train_test_split(train_paths, test_size=0.15, random_state=42)

    train_ds = EEGDataset(train_sub)
    val_ds   = EEGDataset(val_sub)

    _probe = torch.load(train_sub[0], weights_only=True)
    n_ch   = _probe["data"].shape[0]

    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"],
        sampler=make_weighted_sampler(train_ds.labels),
        num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg["batch_size"],
        shuffle=False, num_workers=2, pin_memory=True,
    )

    model = EEGNetBiLSTM(
        n_channels   = n_ch,
        n_timepoints = cfg["n_timepoints"],
        F1           = cfg["F1"],
        D            = cfg["D"],
        dropout      = cfg["dropout"],
        lstm_hidden  = cfg["lstm_hidden"],
        lstm_layers  = cfg["lstm_layers"],
        lstm_dropout = cfg["lstm_dropout"],
        attn_heads   = cfg.get("attn_heads", 4),
        attn_dim     = cfg.get("attn_dim", 16),
        attn_dropout = cfg.get("attn_dropout", 0.1),
    ).to(DEVICE)

    criterion   = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer   = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    scheduler   = torch.optim.lr_scheduler.ReduceLROnPlateau(
                      optimizer, mode="min", patience=3, factor=0.5)
    scaler      = torch.amp.GradScaler('cuda')
    accum_steps = max(1, cfg["effective_batch"] // cfg["batch_size"])

    best_val_loss, best_state, patience_ctr = float("inf"), None, 0

    history = {
        "train_loss": [], "train_acc": [],
        "val_loss":   [], "val_acc":   [],
        "val_sens":   [], "val_spec":  [],
        "lr":         [],
    }

    for epoch in range(cfg["max_epochs"]):
        # ---- Train ----
        model.train()
        train_loss = 0.0
        preds_train, targets_train = [], []
        optimizer.zero_grad()

        for i, (x, y) in enumerate(train_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss   = criterion(logits, y) / accum_steps

            scaler.scale(loss).backward()
            train_loss += loss.item() * accum_steps
            preds_train.extend(logits.detach().argmax(1).cpu().numpy())
            targets_train.extend(y.cpu().numpy())

            if (i + 1) % accum_steps == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

        train_loss /= len(train_loader)
        train_acc   = accuracy_score(targets_train, preds_train)

        # ---- Validate ----
        model.eval()
        val_loss = 0.0
        preds_val, targets_val = [], []

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.amp.autocast('cuda'):
                    logits    = model(x)
                    val_loss += criterion(logits, y).item()
                preds_val.extend(logits.argmax(1).cpu().numpy())
                targets_val.extend(y.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc   = accuracy_score(targets_val, preds_val)
        scheduler.step(val_loss)

        cm_val = confusion_matrix(targets_val, preds_val, labels=[0, 1])
        if cm_val.size == 4:
            tn, fp, fn, tp = cm_val.ravel()
        else:
            tn, fp, fn, tp = 0, 0, 0, 0
        val_sens = tp / (tp + fn + 1e-8)
        val_spec = tn / (tn + fp + 1e-8)

        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_sens"].append(val_sens)
        history["val_spec"].append(val_spec)
        history["lr"].append(current_lr)

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            best_state    = copy.deepcopy(model.state_dict())
            patience_ctr  = 0
        else:
            patience_ctr += 1

        print(
            f"  Ep {epoch+1:02d} | "
            f"TrLoss {train_loss:.4f} TrAcc {train_acc:.4f} | "
            f"VaLoss {val_loss:.4f} VaAcc {val_acc:.4f} | "
            f"Sens {val_sens:.3f} Spec {val_spec:.3f} | "
            f"LR {current_lr:.2e}"
            f"{' ✓' if improved else ''}"
        )

        if patience_ctr >= cfg["patience"]:
            print(f"  Early stop at epoch {epoch+1}")
            break

    if best_state:
        model.load_state_dict(best_state)

    return model, n_ch, history


# ==========================================
# PHASE 2: KD FINE-TUNE ON TARGET PATIENT
# ==========================================
def run_kd_phase2(model, test_patient_id, n_ch, cfg):
    all_files     = [os.path.join(cfg["data_path"], f)
                     for f in os.listdir(cfg["data_path"]) if f.endswith(".pt")]
    patient_files = [f for f in all_files if test_patient_id in f]
    pat_labels    = [1 if "preictal" in f else 0 for f in patient_files]

    support_files, eval_files = train_test_split(
        patient_files, test_size=1.0 - cfg["kd_support_ratio"],
        stratify=pat_labels, random_state=42
    )

    support_ds     = EEGDataset(support_files)
    support_loader = DataLoader(
        support_ds, batch_size=cfg["batch_size"],
        sampler=make_weighted_sampler(support_ds.labels),
        num_workers=2, pin_memory=True,
    )

    p_network = copy.deepcopy(model).to(DEVICE)
    c_network = EEGNetBiLSTM(
        n_channels   = n_ch,
        n_timepoints = cfg["n_timepoints"],
        F1           = cfg["F1"],
        D            = cfg["D"],
        dropout      = cfg["dropout"],
        lstm_hidden  = cfg["lstm_hidden"],
        lstm_layers  = cfg["lstm_layers"],
        lstm_dropout = cfg["lstm_dropout"],
        attn_heads   = cfg.get("attn_heads", 4),
        attn_dim     = cfg.get("attn_dim", 16),
        attn_dropout = cfg.get("attn_dropout", 0.1),
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt_p     = torch.optim.Adam(p_network.parameters(), lr=cfg["lr"] * 0.1)
    opt_c     = torch.optim.Adam(c_network.parameters(), lr=cfg["lr"])
    scaler    = torch.amp.GradScaler('cuda')
    T_temp    = cfg["T_temp"]

    p_network.train()
    c_network.train()

    for epoch in range(cfg["kd_epochs"]):
        ep_loss_p, ep_loss_c = 0.0, 0.0

        for i, (x, y) in enumerate(support_loader):
            x, y = x.to(DEVICE), y.to(DEVICE)

            with torch.amp.autocast('cuda'):
                logits_p = p_network(x)
                logits_c = c_network(x)

                loss_pred_p  = criterion(logits_p, y)
                loss_pred_c  = criterion(logits_c, y)

                log_p_p      = F.log_softmax(logits_p / T_temp, dim=1)
                soft_p_c     = F.softmax(logits_c  / T_temp,    dim=1)
                log_p_c      = F.log_softmax(logits_c / T_temp, dim=1)
                soft_p_p     = F.softmax(logits_p  / T_temp,    dim=1)

                loss_div_p_c = F.kl_div(log_p_p, soft_p_c, reduction='batchmean') * (T_temp ** 2)
                loss_div_c_p = F.kl_div(log_p_c, soft_p_p, reduction='batchmean') * (T_temp ** 2)

                loss_p = loss_pred_p + loss_div_p_c
                loss_c = loss_pred_c + loss_div_c_p

            if i % 2 == 0:
                opt_p.zero_grad()
                scaler.scale(loss_p).backward(retain_graph=True)
                scaler.step(opt_p)
                scaler.update()
            else:
                opt_c.zero_grad()
                scaler.scale(loss_c).backward()
                scaler.step(opt_c)
                scaler.update()

            ep_loss_p += loss_p.item()
            ep_loss_c += loss_c.item()

        print(f"  KD Epoch {epoch+1:02d} | "
              f"P-Loss {ep_loss_p/len(support_loader):.4f} | "
              f"C-Loss {ep_loss_c/len(support_loader):.4f}")

    c_network.eval()
    return c_network, eval_files


# ==========================================
# EVALUATE
# ==========================================
def evaluate(final_model, eval_files, cfg):
    final_model.eval()
    all_probs = []

    with torch.no_grad():
        for fpath in eval_files:
            obj    = torch.load(fpath, weights_only=True)
            x      = obj["data"].unsqueeze(0).to(DEVICE)
            logits = final_model(x)
            prob   = F.softmax(logits, dim=1)[0, 1].item()
            all_probs.append(prob)

    y_true, y_pred, probs_ord = temporal_voting(eval_files, all_probs, cfg)

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, probs_ord)
    except ValueError:
        auc = float("nan")
    cm  = confusion_matrix(y_true, y_pred)

    ta, fa, ih, th, sens = count_alarm_events(
        y_pred, y_true, cfg["window_sec"], cfg["merge_gap_sec"]
    )
    fpr_h = fa / ih if ih > 0 else 0.0

    return {
        "acc": acc, "f1": f1, "auc": auc, "cm": cm,
        "ta": ta, "fa": fa, "ih": ih, "th": th,
        "sens": sens, "fpr_h": fpr_h,
    }


# ==========================================
# FULL LOPO CROSS-VALIDATION LOOP
# ==========================================
save_dir = "/kaggle/working"
os.makedirs(save_dir, exist_ok=True)

all_files = [os.path.join(CFG["data_path"], f)
             for f in os.listdir(CFG["data_path"]) if f.endswith(".pt")]

lopo_results = {}

for fold_idx, test_patient in enumerate(ALL_PATIENTS):
    print(f"\n{'═'*60}")
    print(f"  FOLD {fold_idx+1}/{len(ALL_PATIENTS)} — Test Patient: {test_patient}")
    print(f"  Train Patients: {[p for p in ALL_PATIENTS if p != test_patient]}")
    print(f"{'═'*60}")

    patient_files = [f for f in all_files if test_patient in f]
    if len(patient_files) == 0:
        print(f"  [SKIP] No data found for {test_patient}")
        continue

    # ---- Phase 1 ----
    print(f"\n  --- Phase 1: Training on all patients except {test_patient} ---")
    train_paths = [f for f in all_files if test_patient not in f]
    model, n_ch, history = train_phase1(train_paths, test_patient, CFG)

    torch.save({
        "model_state_dict" : model.state_dict(),
        "cfg"              : CFG,
        "test_patient"     : test_patient,
        "train_patients"   : [p for p in ALL_PATIENTS if p != test_patient],
        "history"          : history,
    }, os.path.join(save_dir, f"phase1_{test_patient}.pt"))
    print(f"  [SAVED] Phase 1 → phase1_{test_patient}.pt")

    # ---- Phase 2 ----
    print(f"\n  --- Phase 2: KD Adaptation for {test_patient} ---")
    final_model, eval_files = run_kd_phase2(model, test_patient, n_ch, CFG)

    torch.save({
        "model_state_dict" : final_model.state_dict(),
        "cfg"              : CFG,
        "test_patient"     : test_patient,
    }, os.path.join(save_dir, f"phase2_{test_patient}.pt"))
    print(f"  [SAVED] Phase 2 → phase2_{test_patient}.pt")

    # ---- Evaluate ----
    print(f"\n  --- Evaluating on {test_patient} held-out set ---")
    if not eval_files:
        print(f"  [SKIP] No eval files for {test_patient}")
        continue

    metrics = evaluate(final_model, eval_files, CFG)
    lopo_results[test_patient] = metrics

    print(f"\n  {'─'*50}")
    print(f"  Results for {test_patient}")
    print(f"  {'─'*50}")
    print(f"  Accuracy              : {metrics['acc']:.4f}")
    print(f"  F1 Score              : {metrics['f1']:.4f}")
    print(f"  AUC-ROC               : {metrics['auc']:.4f}")
    print(f"  CM: TN={metrics['cm'][0,0]} FP={metrics['cm'][0,1]} "
          f"FN={metrics['cm'][1,0]} TP={metrics['cm'][1,1]}")
    print(f"  True Alarms           : {metrics['ta']}")
    print(f"  False Alarms          : {metrics['fa']}")
    print(f"  Sensitivity (Event)   : {metrics['sens']:.4f}")
    print(f"  Interictal Hours      : {metrics['ih']:.2f} h")
    print(f"  Total Hours           : {metrics['th']:.2f} h")
    print(f"  FAR/h                 : {metrics['fpr_h']:.4f}")

    # ---- 3-panel training curves ----
    epochs_ran = len(history["train_loss"])
    ep_axis    = range(1, epochs_ran + 1)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(ep_axis, history["train_loss"], label="Train loss",
                 color="#1f77b4", lw=1.8, marker='o', ms=3)
    axes[0].plot(ep_axis, history["val_loss"],   label="Val loss",
                 color="#ff7f0e", lw=1.8, marker='s', ms=3)
    axes[0].set_title(f"Loss — test: {test_patient}", fontsize=12)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.6)

    axes[1].plot(ep_axis, history["train_acc"], label="Train acc",
                 color="#1f77b4", lw=1.8, marker='o', ms=3)
    axes[1].plot(ep_axis, history["val_acc"],   label="Val acc",
                 color="#ff7f0e", lw=1.8, marker='s', ms=3)
    axes[1].set_title("Accuracy", fontsize=12)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0, 1)
    axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.6)

    axes[2].plot(ep_axis, history["val_sens"], label="Val sensitivity",
                 color="#2ca02c", lw=1.8, marker='^', ms=3)
    axes[2].plot(ep_axis, history["val_spec"], label="Val specificity",
                 color="#d62728", lw=1.8, marker='v', ms=3)
    axes[2].set_title("Val sensitivity vs specificity", fontsize=12)
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Score")
    axes[2].set_ylim(0, 1)
    axes[2].legend(); axes[2].grid(True, linestyle='--', alpha=0.6)

    plt.suptitle(f"Fold {fold_idx+1} — test: {test_patient}", fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"fold_{test_patient}_curves.png"), dpi=100)
    plt.show()


# ==========================================
# AGGREGATE LOPO SUMMARY TABLE
# ==========================================
print(f"\n{'═'*70}")
print(f"  LOPO CROSS-VALIDATION SUMMARY — EEGNet+Attention+LSTM")
print(f"{'═'*70}")
print(f"  {'Patient':<10} {'Acc':>7} {'F1':>7} {'AUC':>7} "
      f"{'TA':>5} {'FA':>5} {'Sens':>7} {'FAR/h':>8}")
print(f"  {'─'*65}")

accs, f1s, aucs, senss, farhs = [], [], [], [], []

for pat, m in lopo_results.items():
    print(f"  {pat:<10} {m['acc']:>7.4f} {m['f1']:>7.4f} {m['auc']:>7.4f} "
          f"{m['ta']:>5} {m['fa']:>5} {m['sens']:>7.4f} {m['fpr_h']:>8.4f}")
    accs.append(m['acc']); f1s.append(m['f1']); aucs.append(m['auc'])
    senss.append(m['sens']); farhs.append(m['fpr_h'])

print(f"  {'─'*65}")
print(f"  {'MEAN':<10} {np.mean(accs):>7.4f} {np.mean(f1s):>7.4f} {np.mean(aucs):>7.4f} "
      f"{'─':>5} {'─':>5} {np.mean(senss):>7.4f} {np.mean(farhs):>8.4f}")
print(f"  {'STD':<10} {np.std(accs):>7.4f} {np.std(f1s):>7.4f} {np.std(aucs):>7.4f} "
      f"{'─':>5} {'─':>5} {np.std(senss):>7.4f} {np.std(farhs):>8.4f}")
print(f"{'═'*70}")

torch.save(lopo_results, os.path.join(save_dir, "lopo_results_all_folds.pt"))
print(f"\n  [SAVED] Full LOPO results → lopo_results_all_folds.pt")